# 01. ArcLLM Core Types

ArcLLM unifies sixteen different LLM providers behind a single set of types. Whatever the
underlying API shape — Anthropic's content blocks, OpenAI's role-tagged messages, Ollama's
raw streams — you build requests and read responses through the same Pydantic models.

This notebook walks through the contract every adapter, module, and agent in the Arc stack
speaks. Get these types right, and routing, caching, audit, telemetry, and policy enforcement
all compose cleanly downstream.

**You will learn:**
- Why a provider-agnostic type layer matters and what it buys you
- How `Message` carries either plain text or a list of typed `ContentBlock`s
- The four content-block variants (`TextBlock`, `ImageBlock`, `ToolUseBlock`, `ToolResultBlock`) and how Pydantic discriminates them
- How `Tool` (definition) and `ToolCall` (invocation) bracket every tool-using turn
- The shape of `LLMResponse` and what `Usage`, `StopReason`, `cost_usd`, and `metadata` actually carry
- How to implement `LLMProvider` so a custom adapter slots into the rest of arcllm

Everything below runs without an API key — every example is pure data construction.

## 1. Setup

Standard Arc walkthrough boilerplate — locate the repo root and put every package's `src/` on `sys.path`.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
_pkgs_root = REPO_ROOT / "packages"
for _pkg in _pkgs_root.iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")
print(f"REPO_ROOT = {REPO_ROOT}")

REPO_ROOT = /Users/joshschultz/Projects/arc/.claude/worktrees/agent-af67a1128a112c184


Pull every public type the notebook will exercise from `arcllm` — these are the
names other Arc packages and your own application code will import.

In [2]:
from arcllm import (
    ContentBlock,
    ImageBlock,
    LLMProvider,
    LLMResponse,
    Message,
    StopReason,
    TextBlock,
    Tool,
    ToolCall,
    ToolResultBlock,
    ToolUseBlock,
    Usage,
)

# Quick sanity check that every symbol resolved.
_loaded = [
    "ContentBlock",
    "ImageBlock",
    "LLMProvider",
    "LLMResponse",
    "Message",
    "StopReason",
    "TextBlock",
    "Tool",
    "ToolCall",
    "ToolResultBlock",
    "ToolUseBlock",
    "Usage",
]
print(f"Loaded {len(_loaded)} symbols from arcllm.")

Loaded 12 symbols from arcllm.


We'll use `claude-sonnet-4-6` as the model identifier in examples — that's the current
default in arcllm's Anthropic adapter catalog. The types themselves are model-agnostic;
the string is just data that flows through `LLMResponse.model`.

In [3]:
DEFAULT_MODEL = "claude-sonnet-4-6"
print(f"Examples will tag responses with model={DEFAULT_MODEL!r}.")

Examples will tag responses with model='claude-sonnet-4-6'.


## 2. Why types matter — the provider-agnostic interface

Every LLM API has its own wire format:

- **Anthropic** uses an array of typed content blocks (`text`, `tool_use`, `tool_result`, ...)
- **OpenAI** uses a flat `content: str` plus a separate `tool_calls` array
- **Google Gemini** uses `parts` keyed by `role`
- **Ollama / vLLM** use OpenAI-compatible payloads but diverge on streaming

If you write your agent against any one of these directly, swapping providers means
rewriting the agent. ArcLLM normalizes the request side (`Message`, `Tool`) and the
response side (`LLMResponse`) so your loop talks to one shape — adapters do the
translation behind the scenes.

Three rules drive the design:

1. **Pydantic v2** — every type is validated at construction. Bad data fails immediately, not three layers deep.
2. **Discriminated unions** — `ContentBlock` uses Pydantic's `Field(discriminator="type")` so dicts off the wire round-trip into the right subclass without manual dispatch.
3. **No over-abstraction** — `Tool.parameters` is a raw `dict[str, Any]` (JSON Schema). The provider expects JSON Schema; we don't wrap it in a custom builder.


In [4]:
# A trivial example of the contract: the same Message shape works for every provider.
user_msg = Message(role="user", content="What's the weather in Austin?")
print(type(user_msg).__name__, user_msg.role, repr(user_msg.content))

Message user "What's the weather in Austin?"


Every type is a Pydantic `BaseModel` (except `LLMProvider`, which is an `ABC`).
That means you get `.model_dump()`, `.model_dump_json()`, and `.model_validate()` for free.

In [5]:
from pydantic import BaseModel

for cls in (
    Message,
    Tool,
    ToolCall,
    TextBlock,
    ImageBlock,
    ToolUseBlock,
    ToolResultBlock,
    Usage,
    LLMResponse,
):
    print(f"{cls.__name__:<18} BaseModel? {issubclass(cls, BaseModel)}")

Message            BaseModel? True
Tool               BaseModel? True
ToolCall           BaseModel? True
TextBlock          BaseModel? True
ImageBlock         BaseModel? True
ToolUseBlock       BaseModel? True
ToolResultBlock    BaseModel? True
Usage              BaseModel? True
LLMResponse        BaseModel? True


## 3. Message — role plus content

A `Message` has two fields:

| Field | Type | Notes |
|---|---|---|
| `role` | `Literal["system", "user", "assistant", "tool"]` | The four roles arcllm normalizes to |
| `content` | `str \| list[ContentBlock]` | A bare string for the simple case; a typed list for tool calls and multi-modal content |

Most user input is a single string. Assistant turns that include tool calls explode into
a list of content blocks so the LLM's structured output survives the round-trip.


In [6]:
# Simple text message — the bare-string content path.
simple = Message(role="user", content="Summarize the last paragraph.")
print(simple.model_dump())

{'role': 'user', 'content': 'Summarize the last paragraph.'}


When the assistant emits both prose and a tool call, you get one `Message` whose
`content` is a list of two blocks. Order is preserved — adapters serialize them in this exact sequence.

In [7]:
assistant_turn = Message(
    role="assistant",
    content=[
        TextBlock(text="Let me check the weather for you."),
        ToolUseBlock(
            id="toolu_weather_1",
            name="get_weather",
            arguments={"city": "Austin", "units": "fahrenheit"},
        ),
    ],
)
for i, block in enumerate(assistant_turn.content):
    print(f"{i}: {type(block).__name__} -> {block.type}")

0: TextBlock -> text
1: ToolUseBlock -> tool_use


Roles are constrained at the type level. Pydantic raises a `ValidationError` if you
try to invent a fifth role — the contract refuses non-canonical values.

In [8]:
from pydantic import ValidationError

try:
    Message(role="admin", content="hack the planet")
except ValidationError as e:
    err = e.errors()[0]
    print(f"rejected: loc={err['loc']}, type={err['type']}")

rejected: loc=('role',), type=literal_error


Empty content lists are intentionally allowed — some providers emit empty assistant
turns mid-stream, and forcing non-empty would mask real protocol cases.

In [9]:
empty = Message(role="assistant", content=[])
print(f"len(content)={len(empty.content)}, valid={isinstance(empty, Message)}")

len(content)=0, valid=True


## 4. ContentBlock hierarchy

`ContentBlock` is a discriminated union of four Pydantic models. Each carries a literal
`type` field that Pydantic uses to pick the right subclass when validating raw dicts.

| Block | `type` literal | Carries |
|---|---|---|
| `TextBlock` | `"text"` | `text: str` |
| `ImageBlock` | `"image"` | `source: str`, `media_type: str` |
| `ToolUseBlock` | `"tool_use"` | `id`, `name`, `arguments: dict[str, Any]` |
| `ToolResultBlock` | `"tool_result"` | `tool_use_id`, `content: str \| list[ContentBlock]` |

`ToolResultBlock.content` is the recursive case — a result block can contain *more*
blocks, which is how rich multi-part tool outputs (text + image, or several text segments) flow back to the model.


In [10]:
# TextBlock — the simplest variant.
text = TextBlock(text="Hello from Claude.")
print(f"type={text.type!r}, text={text.text!r}")

type='text', text='Hello from Claude.'


`ImageBlock` is provider-agnostic on purpose. `source` is whatever the chosen provider
expects — a base64 string for Anthropic inline images, an HTTPS URL for OpenAI image_url
inputs, or a `data:` URI. The adapter rewrites it into the wire format.

In [11]:
image = ImageBlock(
    source="iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNkAAIAAAoAAv/lxKUAAAAASUVORK5CYII=",
    media_type="image/png",
)
print(f"type={image.type!r}, media_type={image.media_type!r}, source_len={len(image.source)}")

type='image', media_type='image/png', source_len=92


`ToolUseBlock` is the structured form of "the model wants to call a tool." Three
required fields: `id` (so the result can be matched back), `name`, and `arguments`
as a fully-parsed dict. Adapter code handles the JSON-string-to-dict parsing for you;
by the time you see a `ToolUseBlock`, arguments are clean.

In [12]:
tool_use = ToolUseBlock(
    id="toolu_01abc",
    name="search_database",
    arguments={"query": "arcllm config", "limit": 10},
)
print(f"id={tool_use.id} name={tool_use.name} arguments={tool_use.arguments}")

id=toolu_01abc name=search_database arguments={'query': 'arcllm config', 'limit': 10}


`ToolResultBlock` closes the tool round-trip. The simple form carries a string;
the rich form carries a list of nested blocks for multi-part output.

In [13]:
result_simple = ToolResultBlock(
    tool_use_id="toolu_01abc",
    content="Found 3 matching records.",
)
print(f"simple -> {type(result_simple.content).__name__}: {result_simple.content!r}")

simple -> str: 'Found 3 matching records.'


In [14]:
result_rich = ToolResultBlock(
    tool_use_id="toolu_01abc",
    content=[
        TextBlock(text="Found 3 records:"),
        TextBlock(text="1. arcllm.config — TOML loader"),
        TextBlock(text="2. arcllm.types — Core types"),
    ],
)
for i, block in enumerate(result_rich.content):
    print(f"{i}: {block.type} -> {block.text}")

0: text -> Found 3 records:
1: text -> 1. arcllm.config — TOML loader
2: text -> 2. arcllm.types — Core types


### How discrimination works

`ContentBlock = Annotated[TextBlock | ImageBlock | ToolUseBlock | ToolResultBlock, Field(discriminator="type")]`

When you hand Pydantic a raw dict with a `type` key, it picks the matching subclass
instead of trying every variant in order. This is much faster *and* gives sharper
validation errors when the dict's shape doesn't match the implied subclass.

In [15]:
from pydantic import TypeAdapter

adapter = TypeAdapter(ContentBlock)

parsed_text = adapter.validate_python({"type": "text", "text": "hello"})
parsed_tool = adapter.validate_python(
    {"type": "tool_use", "id": "c1", "name": "search", "arguments": {}}
)

print(f"text   -> {type(parsed_text).__name__}")
print(f"tool   -> {type(parsed_tool).__name__}")

text   -> TextBlock
tool   -> ToolUseBlock


An unknown `type` is a hard error — adversarial or stale providers can't smuggle
an `audio` block into the model layer.

In [16]:
from pydantic import ValidationError

try:
    adapter.validate_python({"type": "audio", "data": "..."})
except ValidationError as e:
    print(f"rejected: {e.errors()[0]['type']}")

rejected: union_tag_invalid


Discrimination also works *inside* a `Message`. Hand the constructor raw dicts and
Pydantic walks the list, choosing each block's subclass from its `type` field.

In [17]:
msg_from_dicts = Message(
    role="assistant",
    content=[
        {"type": "text", "text": "thinking..."},
        {"type": "tool_use", "id": "c1", "name": "search", "arguments": {"q": "arc"}},
    ],
)
for block in msg_from_dicts.content:
    print(f"{type(block).__name__:<14} type={block.type}")

TextBlock      type=text
ToolUseBlock   type=tool_use


## 5. Tool — the definition you send to the LLM

A `Tool` is the contract you advertise: name, prose description, and the JSON Schema
the LLM must satisfy when it decides to call you.

`Tool.parameters` is `dict[str, Any]` on purpose. Every provider already accepts JSON
Schema; wrapping it in a custom DSL would just add a translation layer that goes stale
every time the spec evolves.

In [18]:
search_tool = Tool(
    name="search_database",
    description="Search the internal knowledge base for relevant documents",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "The search query"},
            "limit": {"type": "integer", "description": "Max results", "default": 10},
        },
        "required": ["query"],
    },
)
print(f"name={search_tool.name}")
print(f"required={search_tool.parameters['required']}")

name=search_database
required=['query']


Tools are designed to round-trip cleanly. `model_dump()` produces a dict that's already
valid for any of arcllm's adapters, and `model_validate()` reads it back into a `Tool`.

In [19]:
import json

tool_dict = search_tool.model_dump()
print(json.dumps(tool_dict, indent=2)[:240], "...")

rebuilt = Tool.model_validate(tool_dict)
assert rebuilt == search_tool, "round-trip failed"
print("round-trip OK")

{
  "name": "search_database",
  "description": "Search the internal knowledge base for relevant documents",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "The  ...
round-trip OK


JSON Schema validation lives at the LLM/tool boundary, not in the `Tool` class itself.
If you need to strictly validate that the parameter schema is well-formed JSON Schema,
compose a third-party validator (`jsonschema`) on top — `Tool` deliberately stays loose.

## 6. ToolCall — the model's invocation request

Where `Tool` is the *definition*, `ToolCall` is the *invocation* — three required fields:

- `id` — provider-assigned correlation ID; matches a `ToolResultBlock.tool_use_id` later
- `name` — which tool to call
- `arguments` — fully-parsed arguments as a dict (no JSON-string indirection)

Two types for two purposes: `ToolUseBlock` lives inside a `Message` (so it survives serialization
as part of the conversation history); `ToolCall` is the standalone parsed entry on `LLMResponse.tool_calls`
(so loop code can iterate without unpacking content blocks).

In [20]:
call = ToolCall(
    id="toolu_01XYZ",
    name="search_database",
    arguments={"query": "federal compliance", "limit": 5},
)
print(f"id={call.id} name={call.name}")
print(f"arguments={call.arguments}")

id=toolu_01XYZ name=search_database
arguments={'query': 'federal compliance', 'limit': 5}


When you persist a conversation for replay or audit, you'll typically convert each
`ToolCall` into a `ToolUseBlock` and embed it inside the assistant `Message`. The
two carry the same fields and the conversion is lossless.

In [21]:
as_block = ToolUseBlock(id=call.id, name=call.name, arguments=call.arguments)
embedded = Message(role="assistant", content=[as_block])
print(embedded.content[0].model_dump())

{'type': 'tool_use', 'id': 'toolu_01XYZ', 'name': 'search_database', 'arguments': {'query': 'federal compliance', 'limit': 5}}


## 7. LLMResponse — the normalized response shape

Whatever provider answered, your loop sees the same `LLMResponse`:

| Field | Type | Meaning |
|---|---|---|
| `content` | `str \| None` | Plain prose response. `None` for pure tool-use turns. |
| `tool_calls` | `list[ToolCall]` | Tools the model wants invoked next. Empty list when there are none. |
| `usage` | `Usage` | Token counts (always populated). |
| `model` | `str` | The model identifier the provider returned. |
| `stop_reason` | `StopReason` | Why generation stopped (see §9). |
| `thinking` | `str \| None` | Extended-thinking output for models that support it. |
| `metadata` | `dict[str, Any] \| None` | Provider-specific extras (request IDs, cache info, ...). |
| `cost_usd` | `float \| None` | Computed cost when the provider catalog has pricing. |
| `raw` | `Any` | The unparsed provider payload. Kept for debugging; excluded from `model_dump`. |


### 7.1 The end-of-turn case

Most responses are a chunk of prose with `stop_reason="end_turn"`. This is what your
loop returns to the user when no more tool calls are needed.

In [22]:
text_response = LLMResponse(
    content="The weather in Austin is 75F and sunny.",
    usage=Usage(input_tokens=500, output_tokens=20, total_tokens=520),
    model=DEFAULT_MODEL,
    stop_reason="end_turn",
)
print(f"stop_reason={text_response.stop_reason}")
print(f"content={text_response.content}")
print(f"tool_calls={text_response.tool_calls}")

stop_reason=end_turn
content=The weather in Austin is 75F and sunny.
tool_calls=[]


### 7.2 The tool-use case

When the model emits tool calls, `content` is `None` (or a short prose preamble depending on
provider) and `tool_calls` carries the parsed list. Your loop iterates `tool_calls`, runs each,
and feeds the results back as `ToolResultBlock`s on the next turn.

In [23]:
tool_response = LLMResponse(
    content=None,
    tool_calls=[
        ToolCall(id="c1", name="get_weather", arguments={"city": "Austin"}),
        ToolCall(id="c2", name="get_time", arguments={"timezone": "CST"}),
    ],
    usage=Usage(input_tokens=800, output_tokens=50, total_tokens=850),
    model=DEFAULT_MODEL,
    stop_reason="tool_use",
)
print(f"calls: {[tc.name for tc in tool_response.tool_calls]}")

calls: ['get_weather', 'get_time']


### 7.3 Mixed prose + tool calls

Anthropic models in particular often emit a short preamble ahead of a tool call. ArcLLM
preserves both — `content` is the prose, `tool_calls` is the structured invocation.

In [24]:
mixed = LLMResponse(
    content="Let me look that up for you.",
    tool_calls=[ToolCall(id="c1", name="search", arguments={"q": "arc"})],
    usage=Usage(input_tokens=50, output_tokens=30, total_tokens=80),
    model=DEFAULT_MODEL,
    stop_reason="tool_use",
)
print(f"content={mixed.content!r}")
print(f"tool_calls={[tc.name for tc in mixed.tool_calls]}")

content='Let me look that up for you.'
tool_calls=['search']


### 7.4 Driving the agent loop

Your loop's branching logic reads off `stop_reason`. Three outcomes cover everything:
1. `end_turn` — return the prose to the user; loop exits.
2. `tool_use` — execute every tool call, append a tool message, ask the model again.
3. anything else — propagate as a controlled error (max-tokens hit, content filter, etc.).

In [25]:
def next_action(response: LLMResponse) -> str:
    """Decide what the loop should do given an LLMResponse."""
    if response.stop_reason == "end_turn":
        return f"return prose to user ({len(response.content or '')} chars)"
    if response.stop_reason == "tool_use":
        return f"run {len(response.tool_calls)} tool call(s) and re-prompt"
    return f"halt — unexpected stop_reason={response.stop_reason}"


for sample in (text_response, tool_response, mixed):
    print(f"{sample.stop_reason:<10} -> {next_action(sample)}")

end_turn   -> return prose to user (39 chars)
tool_use   -> run 2 tool call(s) and re-prompt
tool_use   -> run 1 tool call(s) and re-prompt


### 7.5 Provider extras: `metadata` and `cost_usd`

`metadata` is the escape hatch for provider-specific data that doesn't fit the normalized
schema — Anthropic request IDs, cache control echoes, OpenAI fingerprints, you name it.
`cost_usd` is computed by the adapter when the provider catalog has price-per-token data.

In [26]:
annotated = LLMResponse(
    content="ok",
    usage=Usage(input_tokens=100, output_tokens=20, total_tokens=120),
    model=DEFAULT_MODEL,
    stop_reason="end_turn",
    metadata={
        "anthropic_request_id": "req_018abc",
        "cache_control_echo": True,
    },
    cost_usd=0.00042,
)
print(f"metadata={annotated.metadata}")
print(f"cost_usd={annotated.cost_usd}")

metadata={'anthropic_request_id': 'req_018abc', 'cache_control_echo': True}
cost_usd=0.00042


The `raw` field is the unparsed provider payload. It's `repr=False, exclude=True` so
it never accidentally lands in your audit log or response cache — kept around purely for ad-hoc debugging.

In [27]:
raw_carrier = LLMResponse(
    content="ok",
    usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
    model=DEFAULT_MODEL,
    stop_reason="end_turn",
    raw={"provider_payload": "<truncated 4kb anthropic JSON>"},
)
dumped = raw_carrier.model_dump()
print(f"raw in dumped? {'raw' in dumped}")
print(f"raw on instance? {raw_carrier.raw is not None}")

raw in dumped? False
raw on instance? True


## 8. Usage & cost tracking

`Usage` is the universal token-accounting record. Three required fields cover the
common case; three optional fields handle prompt caching and extended thinking.

| Field | Type | Notes |
|---|---|---|
| `input_tokens` | `int` | Required. Tokens fed to the model. |
| `output_tokens` | `int` | Required. Tokens generated by the model. |
| `total_tokens` | `int` | Required. Sum reported by the provider (we trust them, not arithmetic). |
| `cache_read_tokens` | `int \| None` | Tokens served from prompt cache. `None` if the provider doesn't support caching. |
| `cache_write_tokens` | `int \| None` | Tokens written to prompt cache this turn. |
| `reasoning_tokens` | `int \| None` | Hidden reasoning tokens for extended-thinking models. |


In [28]:
# The plain case — what every provider emits.
basic_usage = Usage(input_tokens=1500, output_tokens=300, total_tokens=1800)
print(f"basic = {basic_usage.model_dump()}")

basic = {'input_tokens': 1500, 'output_tokens': 300, 'total_tokens': 1800, 'cache_read_tokens': None, 'cache_write_tokens': None, 'reasoning_tokens': None}


Caching providers (Anthropic, recently OpenAI) split `input_tokens` between the cache
and the fresh prefix. Extended-thinking models (Claude 3.7+, OpenAI o-series) report
the hidden reasoning load on top.

In [29]:
rich_usage = Usage(
    input_tokens=1500,
    output_tokens=300,
    total_tokens=1800,
    cache_read_tokens=1200,
    cache_write_tokens=300,
    reasoning_tokens=150,
)
print(f"cache hit-rate = {rich_usage.cache_read_tokens / rich_usage.input_tokens:.0%}")
print(f"reasoning load = {rich_usage.reasoning_tokens} tokens")

cache hit-rate = 80%
reasoning load = 150 tokens


Zero-token responses are valid — providers occasionally emit them on retry or filter cases.
Don't over-engineer accounting code to assume tokens are always positive.

In [30]:
zero = Usage(input_tokens=0, output_tokens=0, total_tokens=0)
print(f"zero usage is valid: {zero.model_dump()}")

zero usage is valid: {'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'cache_read_tokens': None, 'cache_write_tokens': None, 'reasoning_tokens': None}


## 9. StopReason — the normalized terminator

`StopReason` is a `Literal` type alias, **not** an enum:

```python
StopReason = Literal["end_turn", "tool_use", "max_tokens", "stop_sequence", "content_filter"]
```

Adapters normalize provider-specific terminators down to one of these five values:

| Value | When |
|---|---|
| `end_turn` | Model is done generating. Default for plain text turns. |
| `tool_use` | Model wants tools invoked before continuing. |
| `max_tokens` | Hit the configured token cap; output may be truncated. |
| `stop_sequence` | Hit a configured stop string. |
| `content_filter` | Provider safety system blocked output. |


In [31]:
from typing import get_args

valid_reasons = get_args(StopReason)
print(f"StopReason values: {valid_reasons}")
print(f"count: {len(valid_reasons)}")

StopReason values: ('end_turn', 'tool_use', 'max_tokens', 'stop_sequence', 'content_filter')
count: 5


Because it's a `Literal`, you typecheck against it as a string — and Pydantic still
rejects bad values at construction time.

In [32]:
from pydantic import ValidationError

for reason in valid_reasons:
    rsp = LLMResponse(
        content="x",
        usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
        model=DEFAULT_MODEL,
        stop_reason=reason,
    )
    print(f"  ok: {rsp.stop_reason}")

try:
    LLMResponse(
        content="x",
        usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
        model=DEFAULT_MODEL,
        stop_reason="self_destruct",  # not in StopReason
    )
except ValidationError as e:
    print(f"rejected invalid stop_reason: {e.errors()[0]['type']}")

  ok: end_turn
  ok: tool_use
  ok: max_tokens
  ok: stop_sequence
  ok: content_filter
rejected invalid stop_reason: literal_error


## 10. LLMProvider — the adapter contract

`LLMProvider` is the abstract base class every adapter (Anthropic, OpenAI, Ollama, ...)
satisfies. Three things to remember:

1. **It's an `abc.ABC`, not a Pydantic model.** Adapters often hold transient state — HTTP
   clients, vault tokens, rate limiters — that doesn't fit the data-class shape.
2. **`name` and `model_name` are required read-only properties** (`@property @abstractmethod`),
   not plain class/instance attributes. arcllm's registry uses `name` to look up adapters by
   provider key (`"anthropic"`, `"openai"`, ...); `model_name` is the resolved model this
   provider instance targets. A subclass must define both as `@property` — a bare
   `self.name = "mock"` assignment inside `__init__` does **not** satisfy an abstract
   property (the ABC check runs against the class at instantiation time, before `__init__`
   ever executes).
3. The rest of the contract is two abstract methods plus one optional concrete one:

   - `async def invoke(messages, tools=None, **kwargs) -> LLMResponse` — make the call.
   - `def validate_config() -> bool` — verify env vars / API keys before runtime.
   - `async def close() -> None` — release HTTP clients (no-op default).

If you want your own provider in the system, subclass and implement these.


In [33]:
import inspect

_is_abc = inspect.isclass(LLMProvider) and bool(getattr(LLMProvider, "__abstractmethods__", None))
print(f"LLMProvider is ABC: {_is_abc}")
print(f"abstract methods: {sorted(LLMProvider.__abstractmethods__)}")

LLMProvider is ABC: True
abstract methods: ['invoke', 'model_name', 'name', 'validate_config']


Trying to instantiate the ABC directly fails — Python won't let you skip the abstract methods.

In [34]:
try:
    LLMProvider()  # type: ignore[abstract]
except TypeError as e:
    print(f"refused: {e}")

refused: Can't instantiate abstract class LLMProvider without an implementation for abstract methods 'invoke', 'model_name', 'name', 'validate_config'


Here's a minimal in-memory provider — useful for tests and walkthroughs. It returns a
canned `LLMResponse`, validates trivially, and has nothing to close.

In [35]:
class MockProvider(LLMProvider):
    """Returns a fixed LLMResponse — handy for tests."""

    def __init__(self, model: str = DEFAULT_MODEL) -> None:
        self.model = model
        self.invocations: list[list[Message]] = []

    @property
    def name(self) -> str:
        return "mock"

    @property
    def model_name(self) -> str:
        return self.model

    async def invoke(self, messages, tools=None, **kwargs):
        self.invocations.append(messages)
        return LLMResponse(
            content="(mock response)",
            usage=Usage(input_tokens=10, output_tokens=5, total_tokens=15),
            model=self.model,
            stop_reason="end_turn",
        )

    def validate_config(self) -> bool:
        return True


mock = MockProvider()
print(f"name={mock.name} model={mock.model} validate_config()={mock.validate_config()}")

name=mock model=claude-sonnet-4-6 validate_config()=True


Driving the mock provider is a one-liner. Note `invoke` is `async` — every adapter in
arcllm is async by design (uvloop + connection-pool friendly).

In [36]:
# Top-level `await` works in IPython kernels — no need for asyncio.run().
_demo_msgs = [Message(role="user", content="hi")]
result = await mock.invoke(_demo_msgs)
print(f"content={result.content}")
print(f"usage={result.usage.model_dump()}")
print(f"recorded_invocations={len(mock.invocations)}")

content=(mock response)
usage={'input_tokens': 10, 'output_tokens': 5, 'total_tokens': 15, 'cache_read_tokens': None, 'cache_write_tokens': None, 'reasoning_tokens': None}
recorded_invocations=1


Subclasses can also override `close()` when they hold real HTTP clients. The default
is an explicit no-op so synthetic adapters don't need to write boilerplate.

In [37]:
await mock.close()
print("close() returned cleanly (default no-op).")

close() returned cleanly (default no-op).


## 11. Serialization round-trips

Every type round-trips through `model_dump()` / `model_validate()` and `model_dump_json()` /
`model_validate_json()`. This is what makes audit storage, replay, and inter-process
transport painless.

In [38]:
rich_msg = Message(
    role="assistant",
    content=[
        TextBlock(text="Here is a tool call"),
        ToolUseBlock(id="c1", name="calc", arguments={"expr": "2+2"}),
    ],
)

as_dict = rich_msg.model_dump()
rebuilt = Message.model_validate(as_dict)
assert rebuilt == rich_msg, "dict round-trip failed"
print("dict round-trip OK")

dict round-trip OK


In [39]:
as_json = rich_msg.model_dump_json()
rebuilt_json = Message.model_validate_json(as_json)
assert rebuilt_json == rich_msg, "json round-trip failed"
print(f"json round-trip OK ({len(as_json)} chars)")

json round-trip OK (148 chars)


Even the recursive `ToolResultBlock` round-trips correctly — discrimination applies
to nested blocks, so a tool result containing a list of blocks survives intact.

In [40]:
rich_result = ToolResultBlock(
    tool_use_id="toolu_42",
    content=[
        TextBlock(text="Header"),
        TextBlock(text="Body"),
    ],
)
result_dict = rich_result.model_dump()
rebuilt_result = ToolResultBlock.model_validate(result_dict)
assert rebuilt_result == rich_result
print(f"nested round-trip OK ({len(rich_result.content)} inner blocks)")

nested round-trip OK (2 inner blocks)


## 12. Summary

The arcllm type layer is the contract everything else builds on. You now know:

- **`Message`** carries `role` and `content` (string or `list[ContentBlock]`).
- **`ContentBlock`** is a discriminated union of `TextBlock`, `ImageBlock`, `ToolUseBlock`, and `ToolResultBlock`, keyed on `type`.
- **`Tool`** advertises a callable to the model with raw JSON Schema parameters; **`ToolCall`** is the model's parsed invocation request.
- **`LLMResponse`** is the normalized response: prose, tool calls, usage, model, stop reason, optional thinking, optional metadata, optional cost.
- **`Usage`** tracks input/output/total tokens plus optional cache and reasoning splits.
- **`StopReason`** is a five-value `Literal` your loop dispatches on.
- **`LLMProvider`** is the `ABC` every adapter implements: `invoke`, `validate_config`, optional `close`.

**API surface covered (12 symbols from `arcllm`):**
`ContentBlock`, `ImageBlock`, `LLMProvider`, `LLMResponse`, `Message`, `StopReason`,
`TextBlock`, `Tool`, `ToolCall`, `ToolResultBlock`, `ToolUseBlock`, `Usage`.

**Next:**
- `02-config-loading.ipynb` — how providers are configured via packaged + user-overlay TOML.
- `03-anthropic-adapter.ipynb` — see the contract above implemented against a real provider.
- `04-agentic-loop.ipynb` — wire `load_model` together with modules and run a real loop.
